## This script identifies all reads matching the missing sequences (unique)
- read the all_exact_match.tsv in (`/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/match_missing_sequences/all_exact_match.tsv`)
- get the unique reads (first column)
- store at (`/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/matched_reads/all_exact_matched_reads.tsv`)

In [70]:
import pandas as pd
from Bio import SeqIO
from Bio.SeqIO.QualityIO import FastqGeneralIterator
import yaml
config_path = "../config/config.yml"
with open(config_path, 'r') as ymlfile:
    config = yaml.safe_load(ymlfile)
print(config)

{'general': {'output_dir': '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results'}, 'directories': {'fastq_dir': '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTemp/fastq', 'exact_match_dir': '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/match_missing_sequences'}, 'files': {'missing_sequences': '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/all_missing_sequences.fa', 'exact_match_result': 'all_exact_match.tsv', 'all_matched_reads': '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/match_missing_sequences/matched_missing_reads.fastq', 'corrected_reads_fastq': '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/match_missing_sequences/corr_matched_missing_reads.fastq', 'unique_matched_reads': '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/match

In [66]:
all_exact_matches_file = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/match_missing_sequences/all_exact_match.tsv" # 343 MB 
all_exact_matches_df = pd.read_csv(all_exact_matches_file, sep="\t", header=None)
all_exact_matches_df.columns = ["read", "oligo", "sequence", "match_length"]
all_exact_matches_df



,read,oligo,sequence,match_length
0,@NB501960:812:HH53WAFX5:1:11101:21622:1110 XI:...,>cardiac_neuro_cava_random:REF_PPP2R2A|ENSG000...,ACTGCAATGCTGCTAATGGATTTTGAGTTGTGGGTACCATATACCC...,270M
1,@NB501960:812:HH53WAFX5:1:11101:4019:1255 XI:Z...,>cardiac_neuro_cava_random:REF_ANK2|ENSG000001...,GTTTTTAGTTGTGTTTAAATCAGGAATTTCCTTACTTTTCAAAATT...,270M
2,@NB501960:812:HH53WAFX5:1:11101:3848:1256 XI:Z...,>cardiac_neuro_cava_random:REF_LMNA|ENSG000001...,TCTTAAATTATCTGAATCTCTTCTGAAGACAGACCTATTAGCTTTT...,270M
3,@NB501960:812:HH53WAFX5:1:11101:5659:1395 XI:Z...,>MK:newcore_110746|chr13-80235318+80235587|ref...,GCTTCTAGTCATAGACAATGGCAATTGCAGAGCAAGACTAATAACA...,270M
4,@NB501960:812:HH53WAFX5:1:11101:23682:1535 XI:...,>cardiac_neuro_cava_random:ALT_RAI1|ENSG000001...,GAGCAACGCAGGAGAGCAGAAAGGCGAGTGCCCTGGGCTCAGGGGG...,270M
...,...,...,...,...
1812287,@NB501960:812:HH53WAFX5:4:21612:8584:19852 XI:...,>cardiac_neuro_cava_random:REF_CNN3|ENSG000001...,CTTTACTTTTCTCTGGCCAGTCAAAAAAGGAAAAAGAAAAAAAAAA...,270M
1812288,@NB501960:812:HH53WAFX5:4:21612:23901:19994 XI...,>cardiac_neuro_cava_random:REF_AHDC1|ENSG00000...,GTCATCACAGTTATGCACACCAGTCCTCTCCCCCACTGCAAATATG...,270M
1812289,@NB501960:812:HH53WAFX5:4:21612:18136:20141 XI...,>cardiac_neuro_cava_random:REF_CNN3|ENSG000001...,AAGAAAGGACACATTTTCCCAATAGTGACATCACTAAAAGCAATTT...,270M
1812290,@NB501960:812:HH53WAFX5:4:21612:8167:20283 XI:...,>MK:tile_47747|chr5-87945240+87945510|T-A-1_forw,ATATCAAAACTGATTACTAGCTGAGCTTTTGTGTCCATAAAATAAA...,270M


In [53]:
# make read column to index
all_exact_matches_df = all_exact_matches_df.set_index("read")
all_exact_matches_df
# make dictionary with keys from all_exact_matches_df (keys: read names)
all_exact_matches_reads = all_exact_matches_df.to_dict('index')
# all_exact_matches_reads.keys()
for key, value in all_exact_matches_reads.items():
    print(key, value)
    break

@NB501960:812:HH53WAFX5:1:11101:21622:1110 XI:Z:GAACTGCAGTGCGGA,YI:Z:AAAAAEEEEEEEEEE {'oligo': '>cardiac_neuro_cava_random:REF_PPP2R2A|ENSG00000221914.11|EH38E2618928_fwd_tile1-1_forw', 'match_length': '270M'}


In [29]:
## get all unique reads in that file (@ read name _ XI:... (Barcode))
all_exact_matches_df["read"].value_counts() # 864426



## Get all unique reads in a file with their sequences (fastq)
- in get_reads_fastq.py
    - fastq's are in `/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTemp/fastq/`
    - load all merge_split{0-29}.join.fastq.gz and take the sequences of all reads also in the all_exact_matches_df["read"] column
- check for duplicates in the resulting reads fasta (config["files"]["all_matched_reads"])

In [72]:
# load /data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/match_missing_sequences/matched_missing_reads.fastq
wrong_fastq_file = config["files"]["all_matched_reads"]
# replace > line with line[1:]
with open(wrong_fastq_file, 'r') as infile:
    with open(config["files"]["corrected_reads_fastq"], 'w') as outfile:
        for line in infile:
            if line.startswith(">"):
                line = line[1:]
            outfile.write(line)

In [75]:
# see get_reads_fasta.py (generating the fasta file of the reads)
## here check if there are duplicates
corr_read_fastq = config["files"]["all_matched_reads"] # just temporary
read_fastq = config["files"]["all_matched_reads"]
unique_read_fastq = config["files"]["unique_matched_reads"]
# read fasta file with biopython and check for duplicates
read_fasta_dict = {}
with open(unique_read_fastq, "w") as outfile:
    with open(read_fastq, "r") as handle:
        for title, seq, qual in FastqGeneralIterator(handle):
        # checking for identical sequences
            if seq not in read_fasta_dict.keys():
                read_fasta_dict[seq] = title
                # write to fasta file (without duplicates)
                outfile.write("@%s\n%s\n+\n%s\n" % (title, seq, qual))


## Investigate if there are duplicated sequences in the missing sequences exact tsv (with reverse complement)
- load the file `/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/resources/align_missing_sequences/missing_reference_exact.tsv`
- check for duplicated sequences

In [11]:
missing_seq_reference_file = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/resources/align_missing_sequences/missing_reference_exact.tsv"
missing_seq_ref_df = pd.read_csv(missing_seq_reference_file, sep="\t", header=None)
missing_seq_ref_df.columns = ["header", "sequence"]
missing_seq_ref_df # 10168 
# ## check if there are any duplicated sequence 
# missing_seq_ref_df["sequence"].value_counts() # 10168
## All unique sequences => reads having exact matches of two sequences are not possible

,header,sequence
0,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,AGGACCGGATCAACTGCCGCCCAGGAGCTCTCGTGCATCCACTCTG...
1,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,AGGACCGGATCAACTGGGGGCGTGTGGTGGGTGGGGGGTGGGTGTT...
2,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,AGGACCGGATCAACTCCCCAACCTCTCTCACGTACACCTGCGTGTT...
3,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,AGGACCGGATCAACTTGTTTGTTGTTTTGAGCACGTTTTAACTTTC...
4,cardiac_neuro_cava_random:RERE|ENSG00000142599...,AGGACCGGATCAACTGGGCTTCGAGGTGAAGCCCCCAGAGCTGGAC...
...,...,...
10163,MK:tile_985|chr1-33363966+33364235|LC28t6_revc,TCGGTTCACGCAATGAGGGGGTATGGCAGTGAGGAATGTGTTCTCC...
10164,MK:tile_985|chr1-33363966+33364235|LC28t7_revc,TCGGTTCACGCAATGAGGGGGTATGGCAGTGAGGAATGTGTTCTCC...
10165,MK:tile_985|chr1-33363966+33364235|LC28t9_revc,TCGGTTCACGCAATGAGGGGGTATGGCAGTGAGGAATGTGTTCTCC...
10166,MK:tile_30307|chr3-171305106+171305375|scrambl...,TCGGTTCACGCAATGACTGAATGAAAGTTAAGGATTGAGCTCACTA...
